Write a JSON Schema spec to describe your tool function


In [ ]:
# Step 1: Write a dictionary of keyword args for your function with sample data

def process_data(ids, profile, primary_id,value):
    pass

# Example Call
process_data(ids=[1, 2, 3], profile={'active': True}, primary_id=1, value=100)

{
  "ids": [1, 2, 3],
  "profile": {  "active": True },
  "primary_id": 1,  
  "value": 100
}


# Step 2: Convert the dictionary to a JSON

{
  "ids": [1, 2, 3],
  "profile": {  
      "active": true 
},
  "primary_id": 1,  
  "value": 100
}

# Step 3: Convert this JSON to JSON SChema using any online converter tool.

{
  "type": "object",
  "properties": {
    "ids": {
      "type": "array",
      "items": [
        {
          "type": "integer"
        },
        {
          "type": "integer"
        },
        {
          "type": "integer"
        }
      ]
    },
    "profile": {
      "type": "object",
      "properties": {
        "active": {
          "type": "boolean"
        }
      },
      "required": [
        "active"
      ]
    },
    "primary_id": {
      "type": "integer"
    },
    "value": {
      "type": "integer"
    }
  },
  "required": [
    "ids",
    "profile",
    "primary_id",
    "value"
  ]
}

# Step 4: Add a description to a each property in the JSON Schema

{
  "type": "object",
  "properties": {
    "ids": {
      "type": "array",
      "description": "A list of integer IDs.",
      "items": [
        {
          "type": "integer"
        },
        {
          "type": "integer"
        },
        {
          "type": "integer"
        }
      ]
    },
    "profile": {
      "type": "object",
      "description": "A profile object containing user information.",
      "properties": {
        "active": {
          "type": "boolean"
        }
      },
      "required": [
        "active"
      ]
    },
    "primary_id": {
      "type": "integer",
      "description": "The primary ID for the user."
    },
    "value": {
      "type": "integer",
      "description": "A numerical value associated with the user."
    }
  },
  "required": [
    "ids",
    "profile",
    "primary_id",
    "value"
  ]
}

In [ ]:
from __future__ import annotations

import os
import sys

from anthropic import Anthropic, APIError
from dotenv import load_dotenv
import json

In [ ]:
load_dotenv(override=False)

API_KEY = os.getenv("CLAUDE_API_KEY")
if not API_KEY:
    sys.stderr.write(
        "ERROR: CLAUDE_API_KEY not found. Create a .env file containing:\n"
        "  CLAUDE_API_KEY=sk-ant-...\n"
    )
    sys.exit(1)

In [ ]:
MODEL = os.getenv("CLAUDE_MODEL", "claude-opus-4-5")

client = Anthropic(api_key=API_KEY)


In [ ]:


messages = []
def add_user_message(messages, text):
  if isinstance(text, str):  
     user_message = {"role": "user", "content": [{"type": "text", "text": text}]}
  else:
     user_message = {"role": "user", "content": text}

  messages.append(user_message)

def add_assistant_message(messages, text):
  if isinstance(text, str):
     assistant_message = {"role": "assistant", "content": [{"type": "text", "text": text}]}
  else:
     assistant_message = {"role": "assistant", "content": text}
 
  messages.append(assistant_message)

def chat(messages, tools=None, system=None, temperature=1.0, stop_sequences=None):
  if stop_sequences is None:
    stop_sequences = []

  params = {
    "model": MODEL,
    "messages": messages,
    "max_tokens": 512,
    "temperature": temperature,
  }
  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  if system:
    params["system"] = [{"type": "text", "text": system}]

  if tools is not None:
    params["tools"] = tools
    params["tool_choice"] = {"type": "auto"}

  response = client.messages.create(**params)

  raw_parts = response.content if hasattr(response, "content") else []
  parts = []
  for block in raw_parts:
    if hasattr(block, "model_dump"):
      parts.append(block.model_dump())
    else:
      parts.append(block)

  text_blocks = []
  for block in parts:
    if isinstance(block, dict):
      block_type = block.get("type")
      if block_type == "text" and block.get("text") is not None:
        text_blocks.append(block["text"])
      elif block_type == "tool_use":
        tool_name = block.get("name", "<unknown>")
        tool_input = block.get("input", {})
        text_blocks.append(f"[tool_use name={tool_name} input={tool_input}]")
    elif isinstance(block, str):
      text_blocks.append(block)

  text = "\n".join(text_blocks)


  # Return in the requested (text, parts) tuple form for compatibility
  return text, parts


In [ ]:
from datetime import datetime

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(date_format)


get_current_datetime_schema = {
    "type": "custom",
    "name": "get_current_datetime",
    "description": "Retrieves the current date and time formatted according to the specified format string.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string that specifies the format of the returned date and time, using Python's strftime format codes.",
            }
        },
        "required": ["date_format"],
    },
}

In [30]:
# Handling Tool use responses
messages = []
add_user_message(messages,  "What time is it?")
#answer = chat(messages)

#answer = chat(messages)

# Call chat and unpack (text, parts)
text, parts = chat(messages, tools=[get_current_datetime_schema])

# Add assistant message using the returned parts
#add_assistant_message(messages, parts)
print(messages)
#print(answer)
print (text) 
print (parts)


[{'role': 'user', 'content': [{'type': 'text', 'text': 'What time is it?'}]}]
[tool_use name=get_current_datetime input={'date_format': '%I:%M:%S %p'}]
[{'id': 'toolu_01XhNyGNHQTdN4gPb3my11v4', 'caller': {'type': 'direct'}, 'input': {'date_format': '%I:%M:%S %p'}, 'name': 'get_current_datetime', 'type': 'tool_use'}]


In [32]:
# Running tool functions
messages = []
add_user_message(messages,  "what is the current date and time?")
#answer = chat(messages)

# Call chat and unpack (text, parts)
text, parts = chat(messages, tools=[get_current_datetime_schema])

# Add assistant message using the returned parts
add_assistant_message(messages, parts)
messages

[{'role': 'user',
  'content': [{'type': 'text', 'text': 'what is the current date and time?'}]},
 {'role': 'assistant',
  'content': [{'id': 'toolu_01LLbbb25jkBLY7UvffdmZws',
    'caller': {'type': 'direct'},
    'input': {'date_format': '%Y-%m-%d %H:%M:%S'},
    'name': 'get_current_datetime',
    'type': 'tool_use'}]}]

In [33]:
# Running tool functions

import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
       return get_current_datetime(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

def run_tools(parts):
    tool_requests = [part for part in parts if isinstance(part, dict) and part.get("type") == "tool_use"]
    tool_result_parts = []

    for tool_request in tool_requests:
        tool_use_id = tool_request.get("id")
        tool_name = tool_request.get("name")
        tool_input = tool_request.get("input")

        #print(f"Running tool: {tool_name} with input: {tool_input}")
        try:
            tool_output = run_tool(tool_name, tool_input)
            #print(f"Tool output: {tool_output}")
            tool_result_part = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_part = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(e),
                "is_error": True,
            }

        tool_result_parts.append(tool_result_part)

    return tool_result_parts

run_tools(parts)




[{'type': 'tool_result',
  'tool_use_id': 'toolu_01LLbbb25jkBLY7UvffdmZws',
  'content': '2026-06-27 19:31:45',
  'is_error': False}]

In [37]:
# Sending tool results back to the model

messages = []
add_user_message(messages,  "What is the current date and time?")

# Call chat and unpack (text, parts)
text, parts = chat(messages, tools=[get_current_datetime_schema])

# Must save full parts (includes tool_use blocks) so the API can match tool_result in the next message
add_assistant_message(messages, parts)
messages


[{'role': 'user',
  'content': [{'type': 'text', 'text': 'What is the current date and time?'}]},
 {'role': 'assistant',
  'content': [{'id': 'toolu_018eYNsiM7nPAa8DkgpCnm25',
    'caller': {'type': 'direct'},
    'input': {'date_format': '%Y-%m-%d %H:%M:%S'},
    'name': 'get_current_datetime',
    'type': 'tool_use'}]}]

In [35]:
# Sending tool results back to the model

import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
       return get_current_datetime(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

def run_tools(parts):
    tool_requests = [part for part in parts if isinstance(part, dict) and part.get("type") == "tool_use"]
    tool_result_parts = []

    for tool_request in tool_requests:
        tool_use_id = tool_request.get("id")
        tool_name = tool_request.get("name")
        tool_input = tool_request.get("input")

        #print(f"Running tool: {tool_name} with input: {tool_input}")
        try:
            tool_output = run_tool(tool_name, tool_input)
            #print(f"Tool output: {tool_output}")
            tool_result_part = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_part = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(e),
                "is_error": True,
            }

        tool_result_parts.append(tool_result_part)

    return tool_result_parts

messages

[{'role': 'user', 'content': [{'type': 'text', 'text': 'What time is it?'}]},
 {'role': 'assistant',
  'content': [{'id': 'toolu_01GzArGk8zUaLpd8tYPinNv4',
    'caller': {'type': 'direct'},
    'input': {'date_format': '%I:%M %p'},
    'name': 'get_current_datetime',
    'type': 'tool_use'}]}]

In [41]:
# Sending tool results back to the model

messages = []
add_user_message(messages,  "What is the current date and time?")

# Call chat and unpack (text, parts)
text, parts = chat(messages, tools=[get_current_datetime_schema])

# Must save full parts (includes tool_use blocks) so the API can match tool_result in the next message
add_assistant_message(messages, parts)
messages

add_user_message(messages, run_tools(parts))
messages

text, parts = chat(messages, tools=[get_current_datetime_schema])
text

'The current date and time is **June 27, 2026 at 7:36:06 PM**.'

In [ ]:
messages = []
add_user_message(messages,  "What is 2+2?")

text, parts = chat(messages, tools=[get_current_datetime_schema])

add_assistant_message(messages, parts)

add_user_message(messages, run_tools(parts))
messages

In [ ]:
# Multi-Turn Conversation with Tools

messages = []
def add_user_message(messages, text):
  if isinstance(text, str):  
     user_message = {"role": "user", "content": [{"type": "text", "text": text}]}
  else:
     user_message = {"role": "user", "content": text}

  messages.append(user_message)

def add_assistant_message(messages, text):
  if isinstance(text, str):
     assistant_message = {"role": "assistant", "content": [{"type": "text", "text": text}]}
  else:
     assistant_message = {"role": "assistant", "content": text}
 
  messages.append(assistant_message)

def chat(messages, tools=None, system=None, temperature=1.0, stop_sequences=None):
  if stop_sequences is None:
    stop_sequences = []

  params = {
    "model": MODEL,
    "messages": messages,
    "max_tokens": 512,
    "temperature": temperature,
  }
  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  if system:
    params["system"] = [{"type": "text", "text": system}]

  if tools is not None:
    params["tools"] = tools
    params["tool_choice"] = {"type": "auto"}

  response = client.messages.create(**params)

  raw_parts = response.content if hasattr(response, "content") else []
  parts = []
  for block in raw_parts:
    if hasattr(block, "model_dump"):
      parts.append(block.model_dump())
    else:
      parts.append(block)

  text_blocks = []
  for block in parts:
    if isinstance(block, dict):
      block_type = block.get("type")
      if block_type == "text" and block.get("text") is not None:
        text_blocks.append(block["text"])
      elif block_type == "tool_use":
        tool_name = block.get("name", "<unknown>")
        tool_input = block.get("input", {})
        text_blocks.append(f"[tool_use name={tool_name} input={tool_input}]")
    elif isinstance(block, str):
      text_blocks.append(block)

  text = "\n".join(text_blocks)


  # Return in the requested (text, parts) tuple form for compatibility
  #return text, parts
  return {
    "parts": parts,
    "text": text,
    "stop_reason": response.stop_reason if hasattr(response, "stop_reason") else None   
  }

In [ ]:
# Multi-Turn Conversation with Tools

def run_conversation(messages):
    while True:
        result = chat(messages, tools=[get_current_datetime_schema])
        
        add_assistant_message(messages, result["parts"])
        print("Assistant:", result["text"])

        if result["stop_reason"] != "tool_use":
            break

        tool_result_parts = run_tools(result["parts"])
        add_user_message(messages, tool_result_parts)

    return messages

In [ ]:
# Multi-Turn Conversation with Tools

messages = []
add_user_message(messages, "What date and time is it?")
#add_user_message(messages, "What is 2+2??")
run_conversation(messages)

# Adding Multiple Tools

In [52]:
# Adding Multiple Tools
## Helper Functions

def chat(messages, tools=None, system=None, temperature=1.0, stop_sequences=None):
  if stop_sequences is None:
    stop_sequences = []

  params = {
    "model": MODEL,
    "messages": messages,
    "max_tokens": 512,
    "temperature": temperature,
  }
  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  if system:
    params["system"] = [{"type": "text", "text": system}]

  if tools is not None:
    params["tools"] = tools
    params["tool_choice"] = {"type": "auto"}

  response = client.messages.create(**params)

  raw_parts = response.content if hasattr(response, "content") else []
  parts = []
  for block in raw_parts:
    if hasattr(block, "model_dump"):
      parts.append(block.model_dump())
    else:
      parts.append(block)

  text_blocks = []
  for block in parts:
    if isinstance(block, dict):
      block_type = block.get("type")
      if block_type == "text" and block.get("text") is not None:
        text_blocks.append(block["text"])
      elif block_type == "tool_use":
        tool_name = block.get("name", "<unknown>")
        tool_input = block.get("input", {})
        text_blocks.append(f"[tool_use name={tool_name} input={tool_input}]")
    elif isinstance(block, str):
      text_blocks.append(block)

  text = "\n".join(text_blocks)

  return {
    "parts": parts,
    "text": text,
    "stop_reason": response.stop_reason if hasattr(response, "stop_reason") else None,
  }

## Tool Functions

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29
                if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
                else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(
        f"----\nSetting the following reminder for {timestamp}:\n{content}\n----"
    )

## Schemas — Anthropic API format

get_current_datetime_schema = {
    "type": "custom",
    "name": "get_current_datetime",
    "description": "Retrieves the current date and time formatted according to the specified format string.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string that specifies the format of the returned date and time, using Python's strftime format codes.",
            }
        },
        "required": ["date_format"],
    },
}


add_duration_to_datetime_schema = {
    "type": "custom",
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add. Can be positive (future) or negative (past). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "Time unit: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "Format string for parsing datetime_str, e.g. '%Y-%m-%d'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "type": "custom",
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text to display in the reminder notification.",
            },
            "timestamp": {
                "type": "string",
                "description": "The date and time when the reminder should trigger, as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS).",
            },
        },
        "required": ["content", "timestamp"],
    },
}


In [53]:
# Adding multiple tools to the conversation

import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
       return get_current_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

def run_tools(parts):
    tool_requests = [part for part in parts if isinstance(part, dict) and part.get("type") == "tool_use"]
    tool_result_parts = []

    for tool_request in tool_requests:
        tool_use_id = tool_request.get("id")
        tool_name = tool_request.get("name")
        tool_input = tool_request.get("input")

        #print(f"Running tool: {tool_name} with input: {tool_input}")
        try:
            tool_output = run_tool(tool_name, tool_input)
            #print(f"Tool output: {tool_output}")
            tool_result_part = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_part = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(e),
                "is_error": True,
            }

        tool_result_parts.append(tool_result_part)

    return tool_result_parts

#messages


In [54]:
# Adding multiple tools to the conversation

def run_conversation(messages):
    while True:
        result = chat(
            messages, 
            tools=[get_current_datetime_schema,
                   add_duration_to_datetime_schema,
                   set_reminder_schema
            ],
        )
        
        add_assistant_message(messages, result["parts"])
        print("Assistant:", result["text"])

        if result["stop_reason"] != "tool_use":
            break

        tool_result_parts = run_tools(result["parts"])
        add_user_message(messages, tool_result_parts)

    return messages

In [56]:
# Adding multiple tools to the conversation - Testing

messages = []
add_user_message(messages, "Set a reminder for me to go to Doctor. The appointment is in 7 days  from now and time should be 10 AM")
#add_user_message(messages, "What is 2+2??")
run_conversation(messages)

Assistant: I'll help you set a reminder for your doctor's appointment. First, let me get the current date, and then calculate the date 7 days from now.
[tool_use name=get_current_datetime input={'date_format': '%Y-%m-%d'}]
Assistant: Now let me calculate the date 7 days from now:
[tool_use name=add_duration_to_datetime input={'datetime_str': '2026-06-27', 'duration': 7, 'unit': 'days'}]
Assistant: Now I'll set the reminder for July 4th, 2026 at 10:00 AM:
[tool_use name=set_reminder input={'content': 'Go to Doctor - Appointment Today', 'timestamp': '2026-07-04T10:00:00'}]
----
Setting the following reminder for 2026-07-04T10:00:00:
Go to Doctor - Appointment Today
----
Assistant: I've set a reminder for your doctor's appointment. Here are the details:

| Detail | Information |
|--------|-------------|
| **Reminder** | Go to Doctor - Appointment Today |
| **Date** | Saturday, July 4, 2026 |
| **Time** | 10:00 AM |

You'll receive a notification at 10:00 AM on that day to remind you about

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Set a reminder for me to go to Doctor. The appointment is in 7 days  from now and time should be 10 AM'}]},
 {'role': 'assistant',
  'content': [{'citations': None,
    'text': "I'll help you set a reminder for your doctor's appointment. First, let me get the current date, and then calculate the date 7 days from now.",
    'type': 'text'},
   {'id': 'toolu_01HwNKtLWG2i16ZmHfHFX46o',
    'caller': {'type': 'direct'},
    'input': {'date_format': '%Y-%m-%d'},
    'name': 'get_current_datetime',
    'type': 'tool_use'}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01HwNKtLWG2i16ZmHfHFX46o',
    'content': '2026-06-27',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [{'citations': None,
    'text': 'Now let me calculate the date 7 days from now:',
    'type': 'text'},
   {'id': 'toolu_013RkCxEnK1VgJCJfENPZqX7',
    'caller': {'type': 'direct'},
    'input': {'datetime_str': '

# Batch Tool Use

In [ ]:
# Batch Tool Use - Without batch tool

messages = []
add_user_message(messages, "What is Mar 12, 2026 plus 50 days?Also, what is Mar 12, 2026 plus 100 days?")
#add_user_message(messages, "What is 2+2??")
run_conversation(messages)

In [ ]:
# Tool Schema — Anthropic API format

batch_tool_schema = {
    "type": "custom",
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


In [ ]:
# Batch Tool Use - Running

import json

def run_batch(tool_input):
    batch_outputs = []
    for invocation in tool_input.get("invocations", []):
        tool_name = invocation.get("name")
        arguments_json = invocation.get("arguments", "{}")
        try:
            arguments = json.loads(arguments_json)
        except json.JSONDecodeError as e:
            batch_outputs.append({
                "tool_name": tool_name,
                "error": f"Invalid JSON for arguments: {e}",
                "output": None
            })
            continue

        try:
            output = run_tool(tool_name, arguments)
            batch_outputs.append({
                "tool_name": tool_name,
                "error": None,
                "output": output
            })
        except Exception as e:
            batch_outputs.append({
                "tool_name": tool_name,
                "error": str(e),
                "output": None
            })
    return batch_outputs

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
       return get_current_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

def run_tools(parts):
    tool_requests = [part for part in parts if isinstance(part, dict) and part.get("type") == "tool_use"]
    tool_result_parts = []

    for tool_request in tool_requests:
        tool_use_id = tool_request.get("id")
        tool_name = tool_request.get("name")
        tool_input = tool_request.get("input")

        #print(f"Running tool: {tool_name} with input: {tool_input}")
        try:
            tool_output = run_tool(tool_name, tool_input)
            #print(f"Tool output: {tool_output}")
            tool_result_part = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_part = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(e),
                "is_error": True,
            }

        tool_result_parts.append(tool_result_part)

    return tool_result_parts

#messages

In [ ]:
# Batch Tool Use - Running Conversation

def run_conversation(messages):
    while True:
        result = chat(
            messages, 
            tools=[get_current_datetime_schema,
                   add_duration_to_datetime_schema,
                   set_reminder_schema,
                   batch_tool_schema
            ],
        )
        
        add_assistant_message(messages, result["parts"])
        print("Assistant:", result["text"])

        if result["stop_reason"] != "tool_use":
            break

        tool_result_parts = run_tools(result["parts"])
        add_user_message(messages, tool_result_parts)

    return messages

In [ ]:
# Batch Tool Use - With batch tool

messages = []
add_user_message(messages, "What is Mar 12, 2026 plus 50 days?Also, what is Mar 12, 2026 plus 100 days?")
#add_user_message(messages, "What is 2+2??")
run_conversation(messages)

# Structured Data With Tools

In [ ]:
# Structured Data With Tools

messages = []
def add_user_message(messages, text):
  if isinstance(text, str):  
     user_message = {"role": "user", "content": [{"type": "text", "text": text}]}
  else:
     user_message = {"role": "user", "content": text}

  messages.append(user_message)

def add_assistant_message(messages, text):
  if isinstance(text, str):
     assistant_message = {"role": "assistant", "content": [{"type": "text", "text": text}]}
  else:
     assistant_message = {"role": "assistant", "content": text}
 
  messages.append(assistant_message)

def chat(messages, tools=None, tool_choice="auto", system=None, temperature=1.0, stop_sequences=None):
  if stop_sequences is None:
    stop_sequences = []

  params = {
    "model": MODEL,
    "messages": messages,
    "max_tokens": 512,
    "temperature": temperature,
  }
  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  if system:
    params["system"] = [{"type": "text", "text": system}]

  if tools is not None:
    params["tools"] = tools
    # tool_choice can be "auto", "any", or a specific tool name string
    if tool_choice == "auto":
      params["tool_choice"] = {"type": "auto"}
    elif tool_choice == "any":
      params["tool_choice"] = {"type": "any"}
    else:
      params["tool_choice"] = {"type": "tool", "name": tool_choice}

  response = client.messages.create(**params)

  raw_parts = response.content if hasattr(response, "content") else []
  parts = []
  for block in raw_parts:
    if hasattr(block, "model_dump"):
      parts.append(block.model_dump())
    else:
      parts.append(block)

  text_blocks = []
  for block in parts:
    if isinstance(block, dict):
      block_type = block.get("type")
      if block_type == "text" and block.get("text") is not None:
        text_blocks.append(block["text"])
      elif block_type == "tool_use":
        tool_name = block.get("name", "<unknown>")
        tool_input = block.get("input", {})
        text_blocks.append(f"[tool_use name={tool_name} input={tool_input}]")
    elif isinstance(block, str):
      text_blocks.append(block)

  text = "\n".join(text_blocks)

  return {
    "parts": parts,
    "text": text,
    "stop_reason": response.stop_reason if hasattr(response, "stop_reason") else None   
  }


In [ ]:
# Structured Data With Tools
# Tool Schemas — Anthropic API format

article_details_schema = {
    "type": "custom",
    "name": "article_details",
    "description": "This tool should be called with details about an article. It accepts information about the article's title, author, and related topics.",
    "input_schema": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "The title of the article. Can be left empty.",
            },
            "author": {
                "type": "string",
                "description": "The name of the article's author. Can be left empty.",
            },
            "topics": {
                "type": "array",
                "items": {"type": "string"},
                "description": "A list of topics or categories that the article covers. Can be an empty list.",
            },
        },
    },
}

to_json_schema = {
    "type": "custom",
    "name": "to_json",
    "description": "This tool processes any JSON data and can be used for generating structured content, transforming information, or creating any JSON-based output needed for your task.",
    "input_schema": {
        "type": "object",
        "additionalProperties": True,
    },
}


In [ ]:
# Structured Data With Tools - Testing

# Structured Data With Tools

messages = []

add_user_message(
    messages,
    "Write a one-paragraph scholarly article about computer science. Include a title and author name",
)

result = chat(messages)

add_assistant_message(messages, result["text"])

result["text"]

In [ ]:
# Structured Data With Tools - Testing with article_details tool

messages = []

add_user_message(
    messages,
    f"""
Analyze the article below and extract key data. Then call the article_details tool.
                 
<article_text>
{result["text"]}                 
</article_text>
""",
)

json_result = chat(
    messages, tools=[article_details_schema], tool_choice="article_details"
)

json_result

In [ ]:
# Structrued Data With Tools - Testing with to_json tool

messages = []

add_user_message(
    messages,
    f"""
Analyze the article below and extract key data. Then call the to_json tool.
                 
<article_text>
{result["text"]}                 
</article_text>

When you call to_json, pass in the following structure:
{{
    "title": str # title of the article,
    "author": str # author of the article,
    "topics": List[str] # List of topics mentioned in the article,
    "num_topics: int # Number of topics mentioned
}}
""",
)

flexible_result = chat(messages, tools=[to_json_schema], tool_choice="to_json")
flexible_result